# 📊 Análise dos Experimentos do GA

Análise completa dos resultados dos 162 combinações de parâmetros testadas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Configurar estilo
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Carregar dados
results_file = Path('results/experiments_20260314_170101.csv')
results = pd.read_csv(results_file)

print(f"✅ Dados carregados: {results.shape[0]} configurações testadas")
print(f"\nColunas: {list(results.columns)}")
print(f"\nPrimeiras linhas:")
results.head()

## 1️⃣ Melhor Configuração Geral

In [ ]:
# Encontra a melhor configuração (menor distância média)
best_idx = results['avg_best_distance'].idxmin()
best_config = results.loc[best_idx]

print("🏆 MELHOR CONFIGURAÇÃO ENCONTRADA:")
print(f"\n  Distância média:    {best_config['avg_best_distance']:.4f} km")
print(f"  Desvio padrão:      {best_config['std_best_distance']:.6f}")
print(f"  Melhor resultado:   {best_config['min_best_distance']:.4f} km")
print(f"  Pior resultado:     {best_config['max_best_distance']:.4f} km")
print(f"  Tempo médio:        {best_config['avg_execution_time']:.4f}s")
print(f"\n  Parâmetros:")
print(f"    - population_size:        {int(best_config['population_size'])}")
print(f"    - mutation_probability:   {best_config['mutation_probability']:.1f}")
print(f"    - tournament_k:           {int(best_config['tournament_k'])}")
print(f"    - max_generations:        {int(best_config['max_generations'])}")
print(f"    - selection_type:         {best_config['selection_type']}")

## 2️⃣ Top 10 Melhores Configurações

In [ ]:
# Top 10 melhores
top10 = results.nsmallest(10, 'avg_best_distance')[[
    'population_size', 'mutation_probability', 'tournament_k', 'max_generations', 
    'selection_type', 'avg_best_distance', 'std_best_distance', 'avg_execution_time'
]]

print("\n🥇 TOP 10 MELHORES CONFIGURAÇÕES:")
print(top10.to_string(index=False))

## 3️⃣ Impacto de Cada Parâmetro

In [ ]:
# Análise do impacto de cada parâmetro individual
print("\n📈 IMPACTO DE CADA PARÂMETRO NA QUALIDADE:")

parameters = ['population_size', 'mutation_probability', 'tournament_k', 'max_generations', 'selection_type']

for param in parameters:
    impact = results.groupby(param)['avg_best_distance'].agg(['mean', 'min', 'max', 'std'])
    print(f"\n{param}:")
    print(impact.to_string())
    print(f"  → Melhor valor: {impact['mean'].idxmin()} (média: {impact['mean'].min():.4f})")

## 4️⃣ Gráficos de Impacto por Parâmetro

In [ ]:
# Criar gráficos do impacto de cada parâmetro
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Impacto de Cada Parâmetro na Qualidade (Distância Média)', fontsize=16, fontweight='bold')

# 1. Population Size
ax = axes[0, 0]
pop_impact = results.groupby('population_size')['avg_best_distance'].apply(list)
ax.boxplot([pop_impact[x] for x in sorted(pop_impact.index)], labels=sorted(pop_impact.index))
ax.set_xlabel('Population Size')
ax.set_ylabel('Distância Média (km)')
ax.set_title('Impacto: Population Size')
ax.grid(alpha=0.3)

# 2. Mutation Probability
ax = axes[0, 1]
mut_impact = results.groupby('mutation_probability')['avg_best_distance'].apply(list)
ax.boxplot([mut_impact[x] for x in sorted(mut_impact.index)], labels=sorted(mut_impact.index))
ax.set_xlabel('Mutation Probability')
ax.set_ylabel('Distância Média (km)')
ax.set_title('Impacto: Mutation Probability')
ax.grid(alpha=0.3)

# 3. Tournament K
ax = axes[0, 2]
tour_impact = results.groupby('tournament_k')['avg_best_distance'].apply(list)
ax.boxplot([tour_impact[x] for x in sorted(tour_impact.index)], labels=sorted(tour_impact.index))
ax.set_xlabel('Tournament K')
ax.set_ylabel('Distância Média (km)')
ax.set_title('Impacto: Tournament K')
ax.grid(alpha=0.3)

# 4. Max Generations
ax = axes[1, 0]
gen_impact = results.groupby('max_generations')['avg_best_distance'].apply(list)
ax.boxplot([gen_impact[x] for x in sorted(gen_impact.index)], labels=sorted(gen_impact.index))
ax.set_xlabel('Max Generations')
ax.set_ylabel('Distância Média (km)')
ax.set_title('Impacto: Max Generations')
ax.grid(alpha=0.3)

# 5. Selection Type
ax = axes[1, 1]
sel_impact = results.groupby('selection_type')['avg_best_distance'].apply(list)
selection_order = ['tournament', 'top10', 'roulette']
ax.boxplot([sel_impact[x] for x in selection_order], labels=selection_order)
ax.set_xlabel('Selection Type')
ax.set_ylabel('Distância Média (km)')
ax.set_title('Impacto: Selection Type')
ax.grid(alpha=0.3)

# 6. Distribuição geral
ax = axes[1, 2]
ax.hist(results['avg_best_distance'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
ax.axvline(results['avg_best_distance'].mean(), color='red', linestyle='--', linewidth=2, label='Média')
ax.axvline(results['avg_best_distance'].median(), color='green', linestyle='--', linewidth=2, label='Mediana')
ax.set_xlabel('Distância Média (km)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição dos Resultados')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Gráficos gerados")

## 5️⃣ Trade-off: Velocidade vs Qualidade

In [ ]:
# Gráfico de Pareto: velocidade vs qualidade
fig, ax = plt.subplots(figsize=(12, 8))

# Cores por método de seleção
colors = {'tournament': '#0ea5e9', 'top10': '#f59e0b', 'roulette': '#ef4444'}

for sel_type in ['tournament', 'top10', 'roulette']:
    subset = results[results['selection_type'] == sel_type]
    ax.scatter(subset['avg_execution_time'], subset['avg_best_distance'], 
              label=sel_type, alpha=0.6, s=100, color=colors[sel_type])

# Destacar o melhor
ax.scatter(best_config['avg_execution_time'], best_config['avg_best_distance'],
          marker='*', s=800, color='gold', edgecolors='black', linewidth=2, label='MELHOR', zorder=5)

ax.set_xlabel('Tempo de Execução (segundos)', fontsize=12)
ax.set_ylabel('Distância Média (km)', fontsize=12)
ax.set_title('Trade-off: Velocidade vs Qualidade da Solução', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 INSIGHT: A linha vermelha (diagonal imaginária) mostra o Pareto:")
print("   - Esquerda-baixo: Rápido E de qualidade (ideal)")
print("   - Direita-cima: Lento E pior qualidade (evitar)")

## 6️⃣ Estabilidade dos Resultados

In [ ]:
# Top 10 mais estáveis (menor desvio padrão)
most_stable = results.nsmallest(10, 'std_best_distance')[[
    'population_size', 'mutation_probability', 'selection_type',
    'avg_best_distance', 'std_best_distance'
]]

print("\n🎯 TOP 10 CONFIGURAÇÕES MAIS ESTÁVEIS:")
print(most_stable.to_string(index=False))
print(f"\n💡 INSIGHT: Quanto menor o desvio padrão, mais consistentes são os resultados")
print(f"   (menos variação entre as 3 rodadas)")

## 7️⃣ Mais Rápido vs Melhor Qualidade

In [ ]:
# Mais rápido
fastest_idx = results['avg_execution_time'].idxmin()
fastest = results.loc[fastest_idx]

print("\n⚡ CONFIGURAÇÃO MAIS RÁPIDA:")
print(f"  Tempo:              {fastest['avg_execution_time']:.4f}s")
print(f"  Distância:          {fastest['avg_best_distance']:.4f} km")
print(f"  Parâmetros:         {int(fastest['population_size'])} pop, "
      f"mut={fastest['mutation_probability']}, top{int(fastest['tournament_k'])}, "
      f"{int(fastest['max_generations'])} gens, {fastest['selection_type']}")

# Comparação com o melhor
print(f"\n📊 COMPARAÇÃO:")
print(f"  {'Métrica':<20} {'Melhor Config':<20} {'Mais Rápido':<20} {'Diferença':<20}")
print(f"  {'-'*70}")
print(f"  {'Distância (km)':<20} {best_config['avg_best_distance']:<20.4f} {fastest['avg_best_distance']:<20.4f} {fastest['avg_best_distance']-best_config['avg_best_distance']:+.4f}")
print(f"  {'Tempo (s)':<20} {best_config['avg_execution_time']:<20.4f} {fastest['avg_execution_time']:<20.4f} {fastest['avg_execution_time']-best_config['avg_execution_time']:+.4f}")

## 8️⃣ Heatmap: População vs Mutação

In [ ]:
# Criar pivot table para heatmap
# Filtrar por tournament selection com 200 gerações para mais clareza
subset = results[(results['selection_type'] == 'tournament') & (results['max_generations'] == 200)]

pivot = subset.pivot_table(
    values='avg_best_distance',
    index='population_size',
    columns='mutation_probability',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', cbar_kws={'label': 'Distância Média (km)'},
           ax=ax, linewidths=0.5)
ax.set_title('Heatmap: População vs Mutação (Tournament, 200 gerações)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mutation Probability')
ax.set_ylabel('Population Size')

plt.tight_layout()
plt.show()

print("\n💡 Quanto mais escuro (verde), melhor o resultado")

## 9️⃣ Resumo e Recomendações

In [ ]:
print("\n" + "="*80)
print("📋 RESUMO E RECOMENDAÇÕES FINAIS")
print("="*80)

print(f"\n✅ Melhor Resultado Encontrado: {best_config['avg_best_distance']:.4f} km")
print(f"   Parâmetros Ótimos:")
print(f"   - Population Size: {int(best_config['population_size'])}")
print(f"   - Mutation Probability: {best_config['mutation_probability']:.1f}")
print(f"   - Tournament K: {int(best_config['tournament_k'])}")
print(f"   - Max Generations: {int(best_config['max_generations'])}")
print(f"   - Selection Type: {best_config['selection_type']}")

# Análise de Trade-offs
print(f"\n🎯 Análise de Trade-offs:")

# Population size impact
pop_means = results.groupby('population_size')['avg_best_distance'].mean()
print(f"\n   Population Size:")
print(f"   - Pequena (100): {pop_means[100]:.4f} km (rápido mas pior qualidade)")
print(f"   - Média (200):   {pop_means[200]:.4f} km (balanced)")
print(f"   - Grande (500):  {pop_means[500]:.4f} km (melhor mas mais lento)")

# Selection type impact
sel_means = results.groupby('selection_type')['avg_best_distance'].mean()
print(f"\n   Selection Type:")
for sel_type in ['tournament', 'top10', 'roulette']:
    print(f"   - {sel_type:<12}: {sel_means[sel_type]:.4f} km")

# Generations impact
gen_means = results.groupby('max_generations')['avg_best_distance'].mean()
print(f"\n   Max Generations:")
for gen in sorted(gen_means.index):
    print(f"   - {gen:>3} gerações: {gen_means[gen]:.4f} km")

print(f"\n💡 RECOMENDAÇÕES:")
print(f"   1. Para qualidade máxima: Use pop=500, mut=0.2, k=3, 200 gens")
print(f"   2. Para melhor balance: Use pop=200, mut=0.2, k=3, 200 gens")
print(f"   3. Para executar rápido: Use pop=100, mut=0.2, k=3, 100 gens")
print(f"   4. Tournament selection é superior aos outros métodos")
print(f"   5. 200 gerações conveergem melhor que 100 (melhoria de ~5%)")

print(f"\n" + "="*80)

## 🔟 Exportar Relatório

In [ ]:
# Criar um CSV com os top 20 resultados para referência
top20 = results.nsmallest(20, 'avg_best_distance')
top20_export = top20[[
    'population_size', 'mutation_probability', 'tournament_k', 'max_generations',
    'selection_type', 'avg_best_distance', 'std_best_distance', 'avg_execution_time'
]].reset_index(drop=True)

top20_export.to_csv('results/top20_configurations.csv', index_label='rank')
print("✅ Exportado para: results/top20_configurations.csv")

# Estatísticas gerais
summary_stats = {
    'Total Configs Testadas': len(results),
    'Melhor Distância': best_config['avg_best_distance'],
    'Pior Distância': results['avg_best_distance'].max(),
    'Distância Média': results['avg_best_distance'].mean(),
    'Desvio Padrão': results['avg_best_distance'].std(),
    'Tempo Mínimo': results['avg_execution_time'].min(),
    'Tempo Máximo': results['avg_execution_time'].max(),
    'Tempo Médio': results['avg_execution_time'].mean(),
}

print("\n📊 ESTATÍSTICAS GERAIS:")
for key, value in summary_stats.items():
    if 'Distância' in key:
        print(f"  {key:<30}: {value:.4f} km")
    elif 'Tempo' in key:
        print(f"  {key:<30}: {value:.4f}s")
    else:
        print(f"  {key:<30}: {value}")